In [ ]:
from pathlib import Path

from idchkg import Pipeline, PipelineConfig
from idchkg.viz import to_yfiles_widget, entity_subgraph

In [ ]:
#pipeline = Pipeline(PipelineConfig(backend="anthropic", model="claude-opus-4-8"))
# Local granite via Ollama, processing chunks concurrently across worker threads
# (graph mutation stays single-threaded, so the result is deterministic).
# For real speedup, start Ollama with OLLAMA_NUM_PARALLEL>1.
pipeline = Pipeline(PipelineConfig(backend="ollama", model="granite4.1:8b", concurrency=8))

In [ ]:
# Build a KG from the bundled, interlinked food- & maritime-history samples.
# glob (not rglob) grabs only the top-level docs, skipping the large fetched
# raw/ (Gutenberg) and wiki/ corpora; point at those to scale up.
paths = sorted(Path("examples/corpora/food").glob("*.txt")) + sorted(Path("examples/corpora/maritime").glob("*.txt"))
docs = pipeline.load_documents(paths)
print(f"Loaded {len(docs)} documents:", ", ".join(d.id for d in docs))

result = pipeline.run(docs, verbose=True)
print(result.stats())

In [21]:
kg = result.graph

In [22]:
kg.entities

[Entity(id='ent-coffee--beverage', label='coffee', type='concept', sense='beverage', aliases=[], descriptions=['the drink prepared from roasted coffee seeds'], contexts=[], mention_count=1),
 Entity(id='ent-roasted-seeds-of-the-coffee-plant', label='roasted seeds of the coffee plant', type='concept', sense='', aliases=[], descriptions=[], contexts=[], mention_count=1),
 Entity(id='ent-coffee--plant', label='coffee', type='concept', sense='plant', aliases=[], descriptions=['the shrub of the genus Coffea'], contexts=['botanical'], mention_count=1),
 Entity(id='ent-the-highlands-of-ethiopia', label='the highlands of Ethiopia', type='concept', sense='', aliases=[], descriptions=[], contexts=[], mention_count=1),
 Entity(id='ent-coffee--stimulating-effect', label='coffee', type='concept', sense='stimulating effect', aliases=[], descriptions=[], contexts=[], mention_count=1),
 Entity(id='ent-ethiopia-long-before-it-spread-through-the-arab-world', label='Ethiopia long before it spread through

In [ ]:
# "coffee" lived in examples/corpus; the food/maritime corpora share commodities
# like sugar, spices, tea, and events like the Atlantic trade / VOC.
sugar = kg.find_entity("sugar")
sugar

In [23]:
to_yfiles_widget(kg)

GraphWidget(layout=Layout(height='800px', width='100%'))

In [ ]:
kg.timeline(sugar)

In [15]:
kg.save_turtle("test.ttl")